In [1]:
import pandas as pd 
import numpy as np 

In [2]:
target = pd.read_csv('target.csv')

In [3]:
realisasi = pd.read_csv('realisasi.csv')

In [4]:
var = pd.read_excel('variable.xlsx')

In [5]:
# Kolom periode
period_cols = ['dec-25', 'jan-26', 'feb-26', 'mar-26']

In [6]:
# Daftar kode cabang
kode_cabang = [
    'ID0010023','ID0010178','ID0010041','ID0010036','ID0010179','ID0010296',
    'ID0010410','ID0010295','ID0019030','ID0018023','ID0019235','ID0018205',
    'ID0018204','ID0010535','ID0010642','ID0010648','ID0019029','ID0019183',
    'ID0010541','ID0019154','ID0010450','ID0018036','ID0010315','ID0010576',
    'ID0010326','ID0018049','ID0010016','ID0010123','ID0018048','ID0010368',
    'ID0010462','ID0010377','ID0010532','ID0019007','ID0010382','ID0018009',
    'ID0010124','ID0019185','ID0010254','ID0010126','ID0010415','ID0010125',
    'ID0018332','ID0010479','ID0010287','ID0018217','ID0010636','ID0010066',
    'ID0010286','ID0018087','ID0019120','ID0010155','ID0010026','ID0010370',
    'ID0010548','ID0010491','ID0019110','ID0019233','ID0010242','ID0010403',
    'ID0018037','ID0010299','ID0019126','ID0010447','ID0018206','ID0019232',
    'ID0010525','ID0010156'
]

# Nilai jumlah sales
jumlah_sales = [
    6,6,11,12,9,3,3,6,6,4,3,4,4,2,6,5,4,6,8,7,6,7,9,8,6,8,9,9,13,13,
    6,9,6,6,4,8,7,7,7,5,4,7,4,5,4,4,4,7,5,4,3,12,10,8,8,10,10,4,8,4,
    6,5,5,6,4,5,4,6
]

# Buat DataFrame baru
df_sales = pd.DataFrame({
    'Kode Cabang': kode_cabang,
    'var_code': 'JUMLAH_SALES',
    'dec-25': jumlah_sales,
    'jan-26': jumlah_sales,
    'feb-26': jumlah_sales,
    'mar-26': jumlah_sales
})

# Tambahkan ke dataframe realisasi
realisasi = pd.concat([realisasi, df_sales], ignore_index=True)

# Rapikan
realisasi = realisasi.sort_values(
    ['Kode Cabang', 'var_code']
).reset_index(drop=True)


In [7]:
# Mapping KPI gabungan:
# nama KPI baru -> daftar var_code penyusun
combine_mapping = {
    'NOA_PHE': ['NoaPayroll', 'NoaHaji', 'NoaEmas'],
    'PHE_CBR': ['NoaPayroll', 'NoaHaji', 'NoaEmas', 'CBR'],
    'NOA_ALL': ['NoaPayroll', 'NoaHaji', 'NoaEmas', 'CBR', 'CBC'],
    'DPK_TOTAL':['TAB', 'GIRO', 'DEPO']
}

combined_rows = []

for new_var_code, source_vars in combine_mapping.items():
    # Ambil data var_code penyusun
    temp = realisasi[
        realisasi['var_code'].isin(source_vars)
    ].copy()

    # Pastikan numerik
    temp[period_cols] = temp[period_cols].apply(
        pd.to_numeric,
        errors='coerce'
    )

    # Jumlahkan per cabang
    temp = (
        temp.groupby(['Kode Cabang'], as_index=False)[period_cols]
        .sum(min_count=1)
    )

    # Tambahkan var_code baru
    temp['var_code'] = new_var_code

    # Simpan hasil
    combined_rows.append(temp)

# Gabungkan semua KPI baru
df_combined = pd.concat(combined_rows, ignore_index=True)

# Tambahkan ke data asli
realisasi = pd.concat([realisasi, df_combined], ignore_index=True)

# Rapikan
realisasi = realisasi.sort_values(
    ['Kode Cabang', 'var_code']
).reset_index(drop=True)


In [8]:
phe = realisasi[
    realisasi['var_code'] == 'NOA_PHE'
].copy()

# Ambil data JUMLAH_SALES
sales = realisasi[
    realisasi['var_code'] == 'JUMLAH_SALES'
].copy()

# Merge berdasarkan Kode Cabang
ratio = phe.merge(
    sales,
    on='Kode Cabang',
    suffixes=('_phe', '_sales')
)

# Hitung rasio untuk setiap periode
for col in period_cols:
    numerator = pd.to_numeric(
        ratio[f'{col}_phe'],
        errors='coerce'
    )
    denominator = pd.to_numeric(
        ratio[f'{col}_sales'],
        errors='coerce'
    )

    # Hindari pembagian dengan nol
    ratio[col] = np.where(
        denominator == 0,
        np.nan,
        numerator / denominator
    )

# Bentuk dataframe final
ratio_result = ratio[['Kode Cabang'] + period_cols].copy()
ratio_result['var_code'] = 'RASIO_NOA_PHE'

# Urutkan kolom
ratio_result = ratio_result[
    ['Kode Cabang', 'var_code'] + period_cols
]

# Tambahkan ke dataframe realisasi
realisasi = pd.concat(
    [realisasi, ratio_result],
    ignore_index=True
)

# Rapikan
realisasi = realisasi.sort_values(
    ['Kode Cabang', 'var_code']
).reset_index(drop=True)

# Cek hasil
realisasi[
    realisasi['var_code'] == 'RASIO_NOA_PHE'
].head()

,Kode Cabang,var_code,dec-25,jan-26,feb-26,mar-26
33,ID0010016,RASIO_NOA_PHE,2129.666667,2150.666667,2184.666667,2141.666667
71,ID0010023,RASIO_NOA_PHE,2898.000000,2961.000000,3016.666667,3082.666667
109,ID0010026,RASIO_NOA_PHE,1640.600000,1672.300000,1692.900000,1682.300000
147,ID0010036,RASIO_NOA_PHE,2008.000000,2015.083333,2070.500000,2079.666667
185,ID0010041,RASIO_NOA_PHE,3050.090909,3077.363636,3122.000000,3174.636364


In [9]:
# Mapping var_code lama -> var_code baru
growth_mapping = {
    'TAB': 'TAB_GROWTH',
    'ACE': 'ACE_GROWTH',
    'GIRO': 'GIRO_GROWTH',
    'DEPO': 'DEPOSITO_GROWTH',
    'USAK': 'USAK_GROWTH',          # sesuaikan jika DGT berasal dari variabel lain
    'KONSUMER': 'CONSUMER_GROWTH',
    'SME': 'SME_GROWTH',
    'Mikro': 'MIKRO_GROWTH',
    'GPB': 'GPB_GROWTH',
    'NOA_ALL': 'ALL_CUST_GROWTH',     # sesuaikan jika ALL_CUST berasal dari variabel lain
    'CBC': 'NON_INDIVIDU_CUST_GROWTH',  # ganti sesuai rumus sebenarnya
    'PHE_CBR': 'INDIVIDU_CUST_GROWTH',             # ganti sesuai rumus sebenarnya
    'RASIO_NOA_PHE': 'RASIO_NOA_PHE_GROWTH',             # ganti sesuai rumus sebenarnya
}

# 1. Ambil hanya var_code yang ingin dihitung growth
hitung_growth = realisasi[
    realisasi['var_code'].isin(growth_mapping.keys())
].copy()

# 2. Pastikan kolom periode bertipe numerik
hitung_growth[period_cols] = hitung_growth[period_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

# 3. Hitung growth dalam bentuk rasio (bukan persen)
#    Rumus: (current - previous) / previous
#    Contoh: 0.05 = growth 5%
hitung_growth[period_cols] = (
    hitung_growth[period_cols]
    .diff(axis=1)
)

# 4. Ubah var_code menjadi nama KPI growth
hitung_growth['var_code'] = hitung_growth['var_code'].map(growth_mapping)

# 5. Tambahkan ke data realisasi asli
realisasi = pd.concat([realisasi, hitung_growth], ignore_index=True)

# 6. (Opsional) rapikan urutan
realisasi = realisasi.sort_values(
    ['Kode Cabang', 'var_code']
).reset_index(drop=True)

# # Cek hasil
# realisasi[
#     realisasi['var_code'].isin(growth_mapping.values())
# ].head(20)

In [10]:
# ==========================================
# HITUNG GROWTH YTD (dibandingkan dengan dec-25)
# Rumus:
#   nilai_periode - nilai_dec_25
#
# Contoh:
#   dec-25 = 100
#   jan-26 = 120  -> 20
#   feb-26 = 150  -> 50
#   mar-26 = 130  -> 30
# ==========================================

# Mapping var_code lama -> var_code baru
growth_ytd_mapping = {
    'TAB': 'TAB_GROWTH_YTD',
    'ACE': 'ACE_GROWTH_YTD',
    'GIRO': 'GIRO_GROWTH_YTD',
    'DEPO': 'DEPOSITO_GROWTH_YTD',
    'USAK': 'USAK_GROWTH_YTD',
    'KONSUMER': 'CONSUMER_GROWTH_YTD',
    'SME': 'SME_GROWTH_YTD',
    'Mikro': 'MIKRO_GROWTH_YTD',
    'GPB': 'GPB_GROWTH_YTD',
    'NOA_ALL': 'ALL_CUST_GROWTH_YTD',
    'CBC': 'NON_INDIVIDU_CUST_GROWTH_YTD',
    'PHE_CBR': 'INDIVIDU_CUST_GROWTH_YTD',
    'RASIO_NOA_PHE': 'RASIO_NOA_PHE_GROWTH_YTD', 
}

# 1. Ambil hanya var_code yang ingin dihitung
hitung_growth_ytd = realisasi[
    realisasi['var_code'].isin(growth_ytd_mapping.keys())
].copy()

# 2. Pastikan numerik
hitung_growth_ytd[period_cols] = hitung_growth_ytd[period_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

# 3. Ambil nilai baseline dec-25
baseline = hitung_growth_ytd['dec-25']

# 4. Hitung growth YTD = nilai periode - dec-25
for col in period_cols:
    hitung_growth_ytd[col] = hitung_growth_ytd[col] - baseline

# 5. (Opsional) Jika ingin dec-25 = 0
hitung_growth_ytd['dec-25'] = 0

# 6. Ubah var_code menjadi nama KPI baru
hitung_growth_ytd['var_code'] = (
    hitung_growth_ytd['var_code']
    .map(growth_ytd_mapping)
)

# 7. Tambahkan ke data asli
realisasi = pd.concat(
    [realisasi, hitung_growth_ytd],
    ignore_index=True
)

# 8. Rapikan
realisasi = realisasi.sort_values(
    ['Kode Cabang', 'var_code']
).reset_index(drop=True)

# # Cek hasil
# realisasi[
#     realisasi['var_code'].isin(growth_ytd_mapping.values())
# ].head(20)

In [11]:
realisasi

,Kode Cabang,var_code,dec-25,jan-26,feb-26,mar-26
0,ID0010016,ACE,166.9834,NaN,90.371600,163.181500
1,ID0010016,ACE_GROWTH,NaN,NaN,NaN,72.809900
2,ID0010016,ACE_GROWTH_YTD,0.0000,NaN,-76.611800,-3.801900
3,ID0010016,ALL_CUST_GROWTH,NaN,434.000000,555.000000,-132.000000
4,ID0010016,ALL_CUST_GROWTH_YTD,0.0000,434.000000,989.000000,857.000000
...,...,...,...,...,...,...
4347,ID0019235,TAB_GROWTH,NaN,-0.315112,-0.199488,-0.413065
4348,ID0019235,TAB_GROWTH_YTD,0.0000,-0.315112,-0.514600,-0.927664
4349,ID0019235,USAK,1.8530,1.966000,1.929000,2.118000
4350,ID0019235,USAK_GROWTH,NaN,0.113000,-0.037000,0.189000


In [14]:
realisasi_var = realisasi[['var_code']].drop_duplicates()
realisasi_var
# print(realisasi_var.tolist())

,var_code
0,ACE
1,ACE_GROWTH
2,ACE_GROWTH_YTD
3,ALL_CUST_GROWTH
4,ALL_CUST_GROWTH_YTD
...,...
59,TAB_GROWTH
60,TAB_GROWTH_YTD
61,USAK
62,USAK_GROWTH
